In [ ]:
import os 
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv("apikey.env")
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_PROJECT'] = "graph function call"
BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
TRAVILY_API_KEY = os.getenv("TAVILY-API-KEY")
gaode_key = os.getenv("GAODEWEATHER_API_KEY")

In [48]:
model = ChatDeepSeek(api_key=API_KEY, 
                     base_url=BASE_URL,
                     model="deepseek-chat",
                     temperature=0.0)

In [2]:
from langchain_tavily import TavilySearch

os.environ["TAVILY_API_KEY"] = TRAVILY_API_KEY
search_tool = TavilySearch(max_results=3)
tools = [search_tool, ]

In [32]:
from langchain_core.tools import StructuredTool
from datetime import datetime

def get_current_time(dummy: str = None) -> dict:
    "获取当前日期和时间"
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return {"current_time": now}

time_tool = StructuredTool.from_function(
    func=get_current_time,
    name="get_current_time",
    description="如果你想知道当前的日期和时间，请使用这个工具",
)

In [33]:
time_tool

StructuredTool(name='get_current_time', description='如果你想知道当前的日期和时间，请使用这个工具', args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x0000023A101540D0>)

In [102]:
tools = []
tools = [time_tool, search_tool]

In [39]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=model,
    tools=tools,
    prompt="如果你需要获取最新的信息，请你使用工具"
)

In [ ]:
# messages = {"messages": [{"role": "user", "content": "今天是几号,今天南宁的天气怎么样？"}]}
# out = agent.invoke(messages)

# for item in out["messages"]:
#     item.pretty_print()

================================ Human Message =================================

今天是几号,今天南宁的天气怎么样？
================================== Ai Message ==================================

我来帮您查询今天的日期和南宁的天气情况。
Tool Calls:
  get_current_time (call_00_FTvc5VKSTUriAUp2hcjxeA93)
 Call ID: call_00_FTvc5VKSTUriAUp2hcjxeA93
  Args:
================================= Tool Message =================================
Name: get_current_time

{"current_time": "2025-10-10 16:03:24"}
================================== Ai Message ==================================

现在我来查询南宁今天的天气情况：
Tool Calls:
  tavily_search (call_00_p72dF9lLZhL41X0zJPYlNbAO)
 Call ID: call_00_p72dF9lLZhL41X0zJPYlNbAO
  Args:
    query: 南宁今天天气 2025年10月10日
    time_range: day
================================= Tool Message =================================
Name: tavily_search

{"query": "南宁今天天气 2025年10月10日", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.zgjia.com/report/guangxi20251010.html", "title

In [92]:
import requests

def get_adcode(keyword: str, api_key: str = gaode_key) -> dict:
    """
    根据地点关键词获取高德的 adcode
    参数:
        keyword: 地点名称，例如 "成都"
        api_key: 高德开放平台 API key
    返回:
        包含 adcode 的字典，例如 {"adcode": "510100"}
    """
    url = "https://restapi.amap.com/v3/config/district"
    params = {
        "key": api_key,
        "keywords": keyword,
        "subdistrict": 0  # 先不获取下级行政区
    }
    
    try:
        response = requests.get(url, params=params)
        data = response.json()
        
        # 检查请求状态
        if data.get("status") == "1" and data.get("districts"):
            # 返回第一个匹配地区的adcode
            adcode = data["districts"][0]["adcode"]
            return {"adcode": adcode, "keyword": keyword}
        else:
            return {
                "error": data.get("info", "查询失败或未找到地区"),
                "keyword": keyword
            }
    except Exception as e:
        return {"error": str(e), "keyword": keyword}
    
adcode_tool = StructuredTool.from_function(
    func=get_adcode,
    name="get_adcode",
    description="用于查询天气的前置：如果用户需要知道adcode必须首先使用这个工具进行查询"
)


In [103]:
tools.append(adcode_tool)

In [ ]:
# agent = create_react_agent(
#     model=model,
#     tools=tools,
#     prompt="如果你需要获取最新的信息，请你使用工具"
# )

# messages = {"messages": [{"role": "user", "content": "柳州的adcode"}]}
# out = agent.invoke(messages)


# for item in out["messages"]:
#     item.pretty_print()

================================ Human Message =================================

柳州的adcode
================================== Ai Message ==================================
Tool Calls:
  get_adcode (call_00_vQs0Mw7ywG5aEbU2mbOCnokd)
 Call ID: call_00_vQs0Mw7ywG5aEbU2mbOCnokd
  Args:
    keyword: 柳州
================================= Tool Message =================================
Name: get_adcode

{"adcode": "450200", "keyword": "柳州"}
================================== Ai Message ==================================

柳州的adcode是450200。


In [106]:
def get_weather_by_adcode(adcode: str, 
                          api_key: str = gaode_key, 
                          extensions: str ="base") -> dict:
    """
    根据高德行政区 adcode 查询天气信息。
    参数:
        adcode: 行政区划代码
        api_key: 高德开放平台的 API Key
        extensions: 
            - "base" 表示实况天气 
            - "all" 表示预报天气
    返回:
        包含天气信息的字典，例如:
        {"city": "成都", "weather": "多云", "temperature": "22", ...}
    """
    url = "https://restapi.amap.com/v3/weather/weatherInfo"
    params = {
        "key": api_key,
        "city": adcode,
        "extensions": extensions
    }
    
    try:
        response = requests.get(url, params=params)
        data = response.json()
        
        if data.get("status") == "1":
            if extensions == "base":
                # 解析实况天气
                lives = data.get("lives", [])
                if lives:
                    live = lives[0]
                    return {
                        "province": live.get("province"),
                        "city": live.get("city"),
                        "weather": live.get("weather"),
                        "temperature": live.get("temperature"),
                        "winddirection": live.get("winddirection"),
                        "windpower": live.get("windpower"),
                        "humidity": live.get("humidity"),
                        "reporttime": live.get("reporttime")
                    }
            elif extensions == "all":
                # 解析预报天气
                forecasts = data.get("forecasts", [])
                if forecasts:
                    # 这里可以处理预报数据，通常会包含未来几天的天气
                    forecast = forecasts[0]
                    return forecast
            return {"error": "未找到天气数据", "adcode": adcode, "extensions": extensions}
        else:
            return {"error": data.get("info", "查询失败"), "adcode": adcode}
    except Exception as e:
        return {"error": str(e), "adcode": adcode}

get_weather_tool = StructuredTool.from_function(
    func=get_weather_by_adcode,
    name="get_weather_by_adcode",
    description="如果用户需要查询实时的天气，首先需要调用get_adcode工具获得adcode在调用"
)


In [107]:
tools.append(get_weather_tool)

In [108]:
tools

[StructuredTool(name='get_current_time', description='如果你想知道当前的日期和时间，请使用这个工具', args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x0000023A101540D0>),
 TavilySearch(max_results=3, api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None)),
 StructuredTool(name='get_adcode', description='用于查询天气的前置：如果用户需要知道adcode必须首先使用这个工具进行查询', args_schema=<class 'langchain_core.utils.pydantic.get_adcode'>, func=<function get_adcode at 0x0000023A12649510>),
 StructuredTool(name='get_adcode', description='如果用户需要查询实时的天气，首先需要调用get_adcode工具获得adcode', args_schema=<class 'langchain_core.utils.pydantic.get_adcode'>, func=<function get_adcode at 0x0000023A1264A200>),
 StructuredTool(name='get_weather_by_adcode', description='如果用户需要查询实时的天气，首先需要调用get_adcode工具获得adcode在调用', args_schema=<class 'langchain_core.utils.pydantic.get_weather_by_adcode'>, func=<function get_weather_by_adcode at 0x0000023A1264B1C0>)]

In [109]:
agent = create_react_agent(
    model=model,
    tools=tools,
    prompt="如果你需要获取最新的信息，请你使用工具"
)

messages = {"messages": [{"role": "user", "content": "请问，柳州今天的天气怎么样？"}]}
out = agent.invoke(messages)


for item in out["messages"]:
    item.pretty_print()

================================ Human Message =================================

请问，柳州今天的天气怎么样？
================================== Ai Message ==================================

我来帮您查询柳州今天的天气情况。首先获取柳州的地理编码，然后查询天气信息。
Tool Calls:
  get_adcode (call_00_99yr8o1kpCEP0COx4HkcfXEU)
 Call ID: call_00_99yr8o1kpCEP0COx4HkcfXEU
  Args:
    keyword: 柳州
================================= Tool Message =================================
Name: get_adcode

{"adcode": "450200", "keyword": "柳州"}
================================== Ai Message ==================================

现在查询柳州的天气情况：
Tool Calls:
  get_weather_by_adcode (call_00_6tqPCxLumMJJxoQQ70jYNiRg)
 Call ID: call_00_6tqPCxLumMJJxoQQ70jYNiRg
  Args:
    adcode: 450200
================================= Tool Message =================================
Name: get_weather_by_adcode

{"province": "广西", "city": "柳州市", "weather": "晴", "temperature": "33", "winddirection": "西", "windpower": "4", "humidity": "49", "reporttime": "2025-10-10 17:04:57"}
=======